In [ ]:
def main(datasources, start_date, end_date):
    """
    BigAlpha AutoAlpha 公式因子挖掘版。

    论文 AutoAlpha 的完整版本包含大规模公式树遗传搜索、PCA-QD 多样性约束、
    warm start、parent-offspring replacement 以及 learning-to-rank 模型。比赛平台
    每次提交需要在有限资源内直接算出测试区间因子，因此这里实现一个轻量化版本：

    1. 从 1 分钟行情聚合日频量价和盘口特征。
    2. 构造公式树终端变量，并枚举/随机生成 depth 1~3 的公式候选。
    3. 用 2019~测试年前两年训练 IC，测试年前一年验证 IC。
    4. 参考 AutoAlpha 的 PCA-QD，用第一主成分签名过滤高度相似的公式。
    5. 对保留下来的公式因子做轻量 Ridge learning-to-rank 融合。

    返回值必须为 ['date', 'instrument', 'factor'] 三列。
    """
    import random
    import numpy as np
    import pandas as pd
    import dai

    EPS = 1e-8
    RANDOM_SEED = 2026
    INSTRUMENTS_TABLE = "bigalpha_2026_instruments"
    FACTORLIB_TABLE = "bigalpha_2026_factorlib"

    # AutoAlpha 搜索规模。数值越大越接近论文算法，但平台运行时间也越长。
    POPULATION_SIZE = 28
    WARM_START_MULTIPLIER = 4
    DEPTH3_SAMPLE_SIZE = 72
    GENE_POOL_SIZE = 16
    MAX_SELECTED_ALPHAS = 12
    PCA_SIM_THRESHOLD = 0.92
    MIN_TRAIN_IC = 0.0015
    MIN_VALID_IC = -0.004
    MIN_ANCHOR_IMPROVEMENT = 0.0007
    MIN_AUTOALPHA_IMPROVEMENT = 0.0015
    MAX_FACTORLIB_COLS = 32
    MAX_FACTORLIB_SELECTED = 5

    def fmt_start(value):
        return pd.to_datetime(value).strftime("%Y-%m-%d 00:00:00")

    def fmt_end(value):
        return pd.to_datetime(value).strftime("%Y-%m-%d 23:59:59")

    def cs_zscore(s):
        """每日横截面去极值、缺失填充和标准化。"""
        s = pd.to_numeric(s, errors="coerce").replace([np.inf, -np.inf], np.nan)
        if s.notna().sum() < 3:
            return pd.Series(0.0, index=s.index)
        lo = s.quantile(0.01)
        hi = s.quantile(0.99)
        s = s.clip(lo, hi)
        s = s.fillna(s.median()).fillna(0.0)
        std = s.std(ddof=0)
        if not np.isfinite(std) or std <= EPS:
            return pd.Series(0.0, index=s.index)
        return (s - s.mean()) / std

    def finalize_factor(df):
        """把任意连续信号转成平台更稳定的每日横截面排名因子。"""
        out = df[["date", "instrument", "factor"]].copy()
        out["date"] = pd.to_datetime(out["date"]).dt.normalize()
        out["instrument"] = out["instrument"].astype(str)
        out["factor"] = pd.to_numeric(out["factor"], errors="coerce").replace([np.inf, -np.inf], np.nan)
        out["factor"] = out.groupby("date")["factor"].transform(
            lambda s: s.fillna(0.0).rank(method="average", pct=True) * 2.0 - 1.0
        )
        out["factor"] = pd.to_numeric(out["factor"], errors="coerce").replace([np.inf, -np.inf], np.nan)
        out = out.dropna(subset=["factor"])
        return out.sort_values(["date", "instrument"]).reset_index(drop=True)

    def read_stock_pool(sd, ed):
        """读取中证 1000 历史成分股，保证输出只覆盖当日股票池。"""
        pool = dai.query(
            f"SELECT date, instrument FROM {INSTRUMENTS_TABLE}",
            filters={"date": [fmt_start(sd), fmt_end(ed)]},
            compression=True,
        ).df()
        if pool.empty:
            return pd.DataFrame(columns=["date", "instrument"])
        pool = pool.copy()
        pool["date"] = pd.to_datetime(pool["date"]).dt.normalize()
        pool["instrument"] = pool["instrument"].astype(str)
        return pool.drop_duplicates(["date", "instrument"])

    def get_table_columns(table_name):
        """尽量用轻量 schema 查询获取列名；失败时返回空列表。"""
        try:
            schema = dai.query(f"DESCRIBE {table_name}", compression=True).df()
            lower_cols = {str(c).lower(): c for c in schema.columns}
            for key in ["column_name", "name", "field"]:
                if key in lower_cols:
                    return [str(x) for x in schema[lower_cols[key]].dropna().tolist()]
        except Exception:
            pass
        try:
            sample = dai.query(
                f"SELECT * FROM {table_name}",
                filters={"date": [fmt_start(start_date), fmt_end(start_date)]},
                compression=True,
            ).df()
            return [str(c) for c in sample.columns]
        except Exception:
            return []

    def factorlib_candidate_columns():
        """选择一小批已有因子列作为外部候选，避免过多列导致提交运行太慢。"""
        table_name = datasources.get("factorlib", FACTORLIB_TABLE)
        cols = get_table_columns(table_name)
        if not cols:
            return []
        excluded = {
            "date",
            "instrument",
            "datetime",
            "symbol",
            "order_book_id",
            "code",
            "name",
            "industry",
            "sector",
        }
        candidates = []
        preferred_tokens = [
            "alpha",
            "factor",
            "mom",
            "rev",
            "vol",
            "liq",
            "turn",
            "size",
            "value",
            "quality",
            "growth",
            "roe",
        ]
        for c in cols:
            lc = c.lower()
            if lc in excluded or lc.startswith("__"):
                continue
            if any(token in lc for token in preferred_tokens):
                candidates.append(c)
        if not candidates:
            candidates = [c for c in cols if c.lower() not in excluded and not c.startswith("__")]
        return candidates[:MAX_FACTORLIB_COLS]

    def quote_identifier(name):
        return '"' + str(name).replace('"', '""') + '"'

    def load_factorlib_panel(sd, ed, pool):
        """
        读取 factorlib 已有因子并对齐股票池。

        factorlib 在不同平台环境下可能是宽表，也可能有部分列不可直接转数值。
        这里全部做容错：读不到或列不合适时返回空特征表，不影响主因子输出。
        """
        table_name = datasources.get("factorlib", FACTORLIB_TABLE)
        cols = factorlib_candidate_columns()
        if not cols or pool.empty:
            return pool[["date", "instrument"]].copy()
        select_cols = ["date", "instrument"] + cols
        sql = "SELECT " + ", ".join(quote_identifier(c) for c in select_cols) + f" FROM {table_name}"
        try:
            raw = dai.query(
                sql,
                filters={"date": [fmt_start(sd), fmt_end(ed)]},
                compression=True,
            ).df()
        except Exception:
            return pool[["date", "instrument"]].copy()
        if raw.empty:
            return pool[["date", "instrument"]].copy()

        raw = raw.copy()
        raw["date"] = pd.to_datetime(raw["date"]).dt.normalize()
        raw["instrument"] = raw["instrument"].astype(str)
        feature_cols = []
        for c in cols:
            if c not in raw.columns:
                continue
            raw[c] = pd.to_numeric(raw[c], errors="coerce").replace([np.inf, -np.inf], np.nan)
            if raw[c].notna().sum() < 100:
                continue
            fcol = "flib_" + str(c)
            raw[fcol] = raw.groupby("date")[c].transform(cs_zscore)
            feature_cols.append(fcol)
        if not feature_cols:
            return pool[["date", "instrument"]].copy()
        raw = raw[["date", "instrument"] + feature_cols].drop_duplicates(["date", "instrument"])
        out = pool[["date", "instrument"]].merge(raw, how="left", on=["date", "instrument"])
        for c in feature_cols:
            out[c] = pd.to_numeric(out[c], errors="coerce").replace([np.inf, -np.inf], np.nan).fillna(0.0)
        return out

    def merge_external_features(panel, sd, ed):
        """把 factorlib 外部因子并入 panel。读不到时保持原 panel 不变。"""
        if panel.empty:
            return panel
        try:
            external = load_factorlib_panel(sd, ed, panel[["date", "instrument"]])
        except Exception:
            return panel
        extra_cols = [c for c in external.columns if c.startswith("flib_")]
        if not extra_cols:
            return panel
        out = panel.merge(external, how="left", on=["date", "instrument"])
        for c in extra_cols:
            out[c] = pd.to_numeric(out[c], errors="coerce").replace([np.inf, -np.inf], np.nan).fillna(0.0)
        return out

    def signal_columns(panel):
        """AutoAlpha 公式树的终端变量：自构造因子 + factorlib 外部因子。"""
        return [c for c in panel.columns if c.startswith("f_") or c.startswith("flib_")]

    def load_daily_panel(sd, ed):
        """
        读取 1 分钟盘口并聚合到日频。

        这里没有硬编码公榜/私榜行情表，而是优先使用平台注入的 datasources["bar1m"]。
        """
        bar1m = datasources.get("bar1m", "bigalpha_2026_stock_bar1m")
        sd_ts = pd.to_datetime(sd).normalize()
        ed_ts = pd.to_datetime(ed).normalize()
        query_start = sd_ts - pd.Timedelta(days=90)

        sql = f"""
        WITH cte AS (
            SELECT
                date,
                instrument,
                strftime(date, '%Y-%m-%d') AS trading_day,
                open,
                high,
                low,
                close,
                volume,
                amount,
                ask_price1,
                bid_price1,
                ask_volume1,
                bid_volume1,
                ask_volume2,
                bid_volume2,
                ask_volume3,
                bid_volume3,
                ask_volume4,
                bid_volume4,
                ask_volume5,
                bid_volume5
            FROM {bar1m}
            WHERE close > 0
        ),
        feat AS (
            SELECT
                *,
                (ask_price1 + bid_price1) / 2 AS mid_price,
                (ask_price1 - bid_price1) / NULLIF((ask_price1 + bid_price1) / 2, 0) AS rel_spread,
                (
                    COALESCE(bid_volume1, 0) + COALESCE(bid_volume2, 0) + COALESCE(bid_volume3, 0)
                    - COALESCE(ask_volume1, 0) - COALESCE(ask_volume2, 0) - COALESCE(ask_volume3, 0)
                ) / NULLIF(
                    COALESCE(bid_volume1, 0) + COALESCE(bid_volume2, 0) + COALESCE(bid_volume3, 0)
                    + COALESCE(ask_volume1, 0) + COALESCE(ask_volume2, 0) + COALESCE(ask_volume3, 0),
                    0
                ) AS depth_imbalance,
                (
                    COALESCE(bid_volume1, 0)
                    + COALESCE(bid_volume2, 0) * EXP(-0.3)
                    + COALESCE(bid_volume3, 0) * EXP(-0.6)
                    + COALESCE(bid_volume4, 0) * EXP(-0.9)
                    + COALESCE(bid_volume5, 0) * EXP(-1.2)
                    - COALESCE(ask_volume1, 0)
                    - COALESCE(ask_volume2, 0) * EXP(-0.3)
                    - COALESCE(ask_volume3, 0) * EXP(-0.6)
                    - COALESCE(ask_volume4, 0) * EXP(-0.9)
                    - COALESCE(ask_volume5, 0) * EXP(-1.2)
                ) / NULLIF(
                    COALESCE(bid_volume1, 0)
                    + COALESCE(bid_volume2, 0) * EXP(-0.3)
                    + COALESCE(bid_volume3, 0) * EXP(-0.6)
                    + COALESCE(bid_volume4, 0) * EXP(-0.9)
                    + COALESCE(bid_volume5, 0) * EXP(-1.2)
                    + COALESCE(ask_volume1, 0)
                    + COALESCE(ask_volume2, 0) * EXP(-0.3)
                    + COALESCE(ask_volume3, 0) * EXP(-0.6)
                    + COALESCE(ask_volume4, 0) * EXP(-0.9)
                    + COALESCE(ask_volume5, 0) * EXP(-1.2),
                    0
                ) AS weighted_imbalance
            FROM cte
            WHERE ask_price1 > 0 AND bid_price1 > 0
        ),
        feat2 AS (
            SELECT
                *,
                (weighted_imbalance * weighted_imbalance)
                / (SQRT(ABS(rel_spread)) + 1e-8) AS raw_pressure
            FROM feat
        ),
        daily AS (
            SELECT
                trading_day,
                instrument,
                ARG_MIN(open, date) AS open,
                ARG_MAX(close, date) AS close,
                MAX(high) AS high,
                MIN(low) AS low,
                SUM(volume) AS volume,
                SUM(amount) AS amount,
                AVG(depth_imbalance) AS avg_depth_imbalance,
                AVG(weighted_imbalance) AS avg_weighted_imbalance,
                ARG_MAX(weighted_imbalance, date) AS last_weighted_imbalance,
                AVG(raw_pressure) AS raw_pressure_mean,
                ARG_MAX(raw_pressure, date) AS last_raw_pressure,
                nanstd(raw_pressure) AS raw_pressure_std,
                AVG(rel_spread) AS avg_rel_spread
            FROM feat2
            GROUP BY trading_day, instrument
        )
        SELECT
            CAST(trading_day AS DATETIME) AS date,
            instrument,
            open,
            close,
            high,
            low,
            volume,
            amount,
            avg_depth_imbalance,
            avg_weighted_imbalance,
            last_weighted_imbalance,
            raw_pressure_mean,
            last_raw_pressure,
            raw_pressure_std,
            avg_rel_spread
        FROM daily
        """

        raw = dai.query(
            sql,
            filters={"date": [fmt_start(query_start), fmt_end(ed_ts)]},
            compression=True,
        ).df()

        pool = read_stock_pool(sd_ts, ed_ts)
        if pool.empty:
            return pd.DataFrame(columns=["date", "instrument", "factor"])
        if raw.empty:
            pool["factor"] = 0.0
            return pool

        raw = raw.copy()
        raw["date"] = pd.to_datetime(raw["date"]).dt.normalize()
        raw["instrument"] = raw["instrument"].astype(str)
        num_cols = [
            "open",
            "close",
            "high",
            "low",
            "volume",
            "amount",
            "avg_depth_imbalance",
            "avg_weighted_imbalance",
            "last_weighted_imbalance",
            "raw_pressure_mean",
            "last_raw_pressure",
            "raw_pressure_std",
            "avg_rel_spread",
        ]
        for col in num_cols:
            raw[col] = pd.to_numeric(raw[col], errors="coerce").replace([np.inf, -np.inf], np.nan)

        raw = raw.sort_values(["instrument", "date"]).drop_duplicates(["date", "instrument"])
        g = raw.groupby("instrument", group_keys=False)

        # 日频量价特征。
        raw["ret_1"] = g["close"].pct_change()
        raw["ret_3"] = g["close"].pct_change(3)
        raw["ret_5"] = g["close"].pct_change(5)
        raw["ret_10"] = g["close"].pct_change(10)
        raw["ret_20"] = g["close"].pct_change(20)
        raw["intraday_ret"] = raw["close"] / (raw["open"] + EPS) - 1.0
        raw["range"] = raw["high"] / (raw["low"] + EPS) - 1.0
        raw["upper_shadow"] = (raw["high"] - np.maximum(raw["open"], raw["close"])) / (raw["close"].abs() + EPS)
        raw["lower_shadow"] = (np.minimum(raw["open"], raw["close"]) - raw["low"]) / (raw["close"].abs() + EPS)
        raw["vwap"] = raw["amount"] / (raw["volume"] + EPS)
        raw["vwap_gap"] = raw["vwap"] / (raw["close"] + EPS) - 1.0
        raw["amount_log"] = np.log1p(raw["amount"].clip(lower=0))
        raw["volume_log"] = np.log1p(raw["volume"].clip(lower=0))
        raw["amount_ma20"] = g["amount"].transform(lambda s: s.rolling(20, min_periods=8).mean())
        raw["volume_ma20"] = g["volume"].transform(lambda s: s.rolling(20, min_periods=8).mean())
        raw["amount_ratio"] = raw["amount"] / (raw["amount_ma20"] + EPS) - 1.0
        raw["volume_ratio"] = raw["volume"] / (raw["volume_ma20"] + EPS) - 1.0
        raw["volatility_5"] = g["ret_1"].transform(lambda s: s.rolling(5, min_periods=3).std())
        raw["volatility_20"] = g["ret_1"].transform(lambda s: s.rolling(20, min_periods=8).std())
        raw["illiq"] = raw["ret_1"].abs() / (raw["amount"].abs() + EPS)
        raw["book_pressure"] = raw["avg_weighted_imbalance"] / (
            np.sqrt(raw["avg_rel_spread"].abs()) + EPS
        )
        raw["pressure_z"] = (raw["last_raw_pressure"] - raw["raw_pressure_mean"]) / (
            raw["raw_pressure_std"].abs() + EPS
        )

        raw["next_close"] = g["close"].shift(-1)
        raw["label"] = raw["next_close"] / (raw["close"] + EPS) - 1.0

        raw_feature_cols = [
            "ret_1",
            "ret_3",
            "ret_5",
            "ret_10",
            "ret_20",
            "intraday_ret",
            "range",
            "upper_shadow",
            "lower_shadow",
            "vwap_gap",
            "amount_log",
            "volume_log",
            "amount_ratio",
            "volume_ratio",
            "volatility_5",
            "volatility_20",
            "illiq",
            "avg_depth_imbalance",
            "avg_weighted_imbalance",
            "last_weighted_imbalance",
            "raw_pressure_mean",
            "pressure_z",
            "avg_rel_spread",
            "book_pressure",
        ]

        feature_cols = []
        for col in raw_feature_cols:
            fcol = "f_" + col
            raw[col] = pd.to_numeric(raw[col], errors="coerce").replace([np.inf, -np.inf], np.nan)
            raw[fcol] = raw.groupby("date")[col].transform(cs_zscore)
            feature_cols.append(fcol)

        raw["label"] = pd.to_numeric(raw["label"], errors="coerce").replace([np.inf, -np.inf], np.nan)
        raw["label_cs"] = raw.groupby("date")["label"].transform(cs_zscore)
        raw.loc[raw["label"].isna(), "label_cs"] = np.nan

        raw = raw[(raw["date"] >= sd_ts) & (raw["date"] <= ed_ts)]
        keep_cols = ["date", "instrument", "close", "label", "label_cs"] + feature_cols
        panel = pool.merge(raw[keep_cols], how="left", on=["date", "instrument"])
        panel["date"] = pd.to_datetime(panel["date"]).dt.normalize()
        panel["instrument"] = panel["instrument"].astype(str)
        for col in ["close", "label", "label_cs"] + feature_cols:
            panel[col] = pd.to_numeric(panel[col], errors="coerce").replace([np.inf, -np.inf], np.nan)
        for col in feature_cols:
            panel[col] = panel[col].fillna(0.0)
        return panel.sort_values(["date", "instrument"]).reset_index(drop=True)

    def col(name):
        return ("col", name)

    def unary(op, child):
        return ("unary", op, child)

    def binary(op, left, right):
        return ("binary", op, left, right)

    def linear(terms):
        return ("linear", tuple(terms))

    def stable_base_formula():
        """此前更稳的盘口基线公式，作为 AutoAlpha 搜索的锚。"""
        return linear(
            (
                (0.40, col("f_avg_depth_imbalance")),
                (-0.25, col("f_avg_rel_spread")),
                (-0.18, col("f_intraday_ret")),
                (-0.10, col("f_range")),
                (0.05, col("f_amount_log")),
            )
        )

    def stable_base_signal(panel):
        """稳定基线信号；任何缺失字段都按 0 处理，保证兜底不报错。"""
        weights = {
            "f_avg_depth_imbalance": 0.40,
            "f_avg_rel_spread": -0.25,
            "f_intraday_ret": -0.18,
            "f_range": -0.10,
            "f_amount_log": 0.05,
        }
        signal = pd.Series(0.0, index=panel.index)
        for name, weight in weights.items():
            if name in panel.columns:
                x = pd.to_numeric(panel[name], errors="coerce").replace([np.inf, -np.inf], np.nan).fillna(0.0)
                signal = signal + weight * x
        return signal

    def anchor_formula_library():
        """
        稳健 anchor 因子库。

        AutoAlpha 论文强调先找到有效 root genes，再围绕这些 genes 搜索更高阶公式。
        这里把已知较稳的盘口/反转/流动性结构作为 root-level anchors，让搜索有更好的起点。
        """
        return [
            (
                "stable_book",
                stable_base_formula(),
            ),
            (
                "weighted_book",
                linear(
                    (
                        (0.28, col("f_avg_depth_imbalance")),
                        (0.18, col("f_avg_weighted_imbalance")),
                        (0.14, col("f_last_weighted_imbalance")),
                        (0.08, col("f_book_pressure")),
                        (-0.22, col("f_avg_rel_spread")),
                        (-0.14, col("f_intraday_ret")),
                        (-0.08, col("f_range")),
                        (0.04, col("f_amount_log")),
                    )
                ),
            ),
            (
                "reversal_book",
                linear(
                    (
                        (0.32, col("f_avg_depth_imbalance")),
                        (-0.16, col("f_ret_1")),
                        (-0.10, col("f_ret_3")),
                        (-0.20, col("f_intraday_ret")),
                        (-0.08, col("f_range")),
                        (-0.18, col("f_avg_rel_spread")),
                        (0.06, col("f_amount_log")),
                    )
                ),
            ),
            (
                "liquidity_book",
                linear(
                    (
                        (0.26, col("f_avg_depth_imbalance")),
                        (0.12, col("f_avg_weighted_imbalance")),
                        (-0.20, col("f_avg_rel_spread")),
                        (-0.10, col("f_volatility_5")),
                        (-0.08, col("f_volatility_20")),
                        (-0.06, col("f_amount_ratio")),
                        (-0.04, col("f_volume_ratio")),
                        (0.05, col("f_amount_log")),
                    )
                ),
            ),
            (
                "shadow_reversal",
                linear(
                    (
                        (0.24, col("f_avg_depth_imbalance")),
                        (-0.18, col("f_intraday_ret")),
                        (-0.12, col("f_upper_shadow")),
                        (0.08, col("f_lower_shadow")),
                        (-0.16, col("f_range")),
                        (-0.18, col("f_avg_rel_spread")),
                        (0.04, col("f_vwap_gap")),
                    )
                ),
            ),
        ]

    def formula_name(formula):
        kind = formula[0]
        if kind == "col":
            return formula[1]
        if kind == "unary":
            return formula[1] + "(" + formula_name(formula[2]) + ")"
        if kind == "binary":
            return "(" + formula_name(formula[2]) + formula[1] + formula_name(formula[3]) + ")"
        if kind == "linear":
            return "linear(" + ",".join(str(round(w, 4)) + "*" + formula_name(f) for w, f in formula[1]) + ")"
        return str(formula)

    def safe_series(values, index):
        s = pd.Series(values, index=index)
        s = pd.to_numeric(s, errors="coerce").replace([np.inf, -np.inf], np.nan)
        return s.clip(-30.0, 30.0)

    def eval_formula(formula, df):
        """解释执行公式树。所有终端变量已做过横截面标准化。"""
        kind = formula[0]
        if kind == "col":
            return df[formula[1]].astype(float)
        if kind == "linear":
            acc = pd.Series(0.0, index=df.index)
            for weight, child in formula[1]:
                acc = acc + float(weight) * eval_formula(child, df)
            return safe_series(acc, df.index)
        if kind == "unary":
            x = eval_formula(formula[2], df)
            op = formula[1]
            if op == "neg":
                y = -x
            elif op == "abs":
                y = x.abs()
            elif op == "square":
                y = np.sign(x) * (x.abs() ** 2)
            elif op == "sqrt":
                y = np.sign(x) * np.sqrt(x.abs())
            elif op == "tanh":
                y = np.tanh(x)
            else:
                y = x
            return safe_series(y, df.index)
        if kind == "binary":
            a = eval_formula(formula[2], df)
            b = eval_formula(formula[3], df)
            op = formula[1]
            if op == "add":
                y = a + b
            elif op == "sub":
                y = a - b
            elif op == "mul":
                y = a * b
            elif op == "div":
                denom = np.where(b.abs() < 0.15, np.nan, b)
                y = a / denom
            elif op == "min":
                y = np.minimum(a, b)
            elif op == "max":
                y = np.maximum(a, b)
            else:
                y = a
            return safe_series(y, df.index)
        return pd.Series(0.0, index=df.index)

    def clean_factor_values(raw_values, dates):
        x = pd.to_numeric(raw_values, errors="coerce").replace([np.inf, -np.inf], np.nan)
        x = x.groupby(dates).transform(cs_zscore)
        return pd.to_numeric(x, errors="coerce").replace([np.inf, -np.inf], np.nan).fillna(0.0)

    def fast_mean_ic(x, y, dates, mask):
        """
        近似日均 IC。x 和 y 都按日标准化后，每日 corr 近似为 mean(x*y)。
        """
        valid = mask & x.notna() & y.notna()
        if int(valid.sum()) < 1000:
            return -999.0
        prod = x.loc[valid] * y.loc[valid]
        ic_by_date = prod.groupby(dates.loc[valid]).mean()
        if len(ic_by_date) == 0:
            return -999.0
        return float(ic_by_date.replace([np.inf, -np.inf], np.nan).dropna().mean())

    def first_pc_signature(formula, df, sample_dates):
        """
        AutoAlpha PCA-QD 的轻量实现：用候选 alpha 在抽样日期上的 date x stock 矩阵，
        通过幂法计算第一主成分，并用主成分相关性近似公式相似度。
        """
        if not sample_dates:
            return None
        raw_values = eval_formula(formula, df)
        x = clean_factor_values(raw_values, df["date"])
        sub = pd.DataFrame(
            {
                "date": df["date"],
                "instrument": df["instrument"],
                "x": x,
            }
        )
        sub = sub[sub["date"].isin(sample_dates)]
        if sub.empty:
            return None
        mat = sub.pivot_table(index="date", columns="instrument", values="x", aggfunc="mean").fillna(0.0)
        if mat.shape[0] < 5 or mat.shape[1] < 20:
            return None
        arr = mat.to_numpy(dtype=np.float64, copy=True)
        arr = arr - arr.mean(axis=0, keepdims=True)
        norm = np.linalg.norm(arr)
        if not np.isfinite(norm) or norm <= EPS:
            return None
        rng = np.random.default_rng(RANDOM_SEED)
        v = rng.normal(size=arr.shape[1])
        v = v / (np.linalg.norm(v) + EPS)
        for _ in range(6):
            v = arr.T @ (arr @ v)
            v = v / (np.linalg.norm(v) + EPS)
        return v

    def corr_signature(sig_a, sig_b):
        if sig_a is None or sig_b is None or len(sig_a) != len(sig_b):
            return 0.0
        a = sig_a - np.mean(sig_a)
        b = sig_b - np.mean(sig_b)
        denom = np.linalg.norm(a) * np.linalg.norm(b)
        if denom <= EPS:
            return 0.0
        return float(np.dot(a, b) / denom)

    def evaluate_candidates(hist_panel, terminal_cols, train_mask, valid_mask):
        dates = hist_panel["date"]
        label = hist_panel["label_cs"]
        cache = {}

        def evaluate(formula):
            key = formula_name(formula)
            if key in cache:
                return cache[key]
            raw_values = eval_formula(formula, hist_panel)
            x = clean_factor_values(raw_values, dates)
            train_ic_raw = fast_mean_ic(x, label, dates, train_mask)
            valid_ic_raw = fast_mean_ic(x, label, dates, valid_mask)
            direction = 1.0 if train_ic_raw >= 0 else -1.0
            oriented_formula = formula if direction > 0 else unary("neg", formula)
            train_ic = abs(train_ic_raw)
            valid_ic = direction * valid_ic_raw
            score = 0.55 * valid_ic + 0.45 * train_ic
            item = {
                "formula": oriented_formula,
                "raw_formula": formula,
                "name": formula_name(oriented_formula),
                "train_ic": train_ic,
                "valid_ic": valid_ic,
                "score": score,
            }
            cache[key] = item
            return item

        # depth 1：枚举原始终端和一元变换，形成第一层 gene pool。
        terminals = [col(c) for c in terminal_cols]
        depth1 = []
        for f in terminals:
            depth1.append(f)
            depth1.append(unary("sqrt", f))
            depth1.append(unary("tanh", f))
            depth1.append(unary("square", f))

        for _, anchor_formula in anchor_formula_library():
            depth1.append(anchor_formula)

        depth1_scores = [evaluate(f) for f in depth1]
        depth1_scores = sorted(depth1_scores, key=lambda x: x["train_ic"], reverse=True)
        gene_pool = [x["formula"] for x in depth1_scores[:GENE_POOL_SIZE]]

        # warm start：生成 K 倍 population 的 depth 2 公式，再择优初始化。
        rng = random.Random(RANDOM_SEED)
        ops = ["add", "sub", "mul", "div", "min", "max"]
        source_pool = gene_pool + terminals
        warm_formulas = []
        for _ in range(POPULATION_SIZE * WARM_START_MULTIPLIER):
            a = rng.choice(source_pool)
            b = rng.choice(source_pool)
            op = rng.choice(ops)
            if formula_name(a) != formula_name(b):
                warm_formulas.append(binary(op, a, b))
        warm_scores = [evaluate(f) for f in warm_formulas]
        warm_scores = sorted(warm_scores, key=lambda x: x["train_ic"], reverse=True)
        population = warm_scores[:POPULATION_SIZE]

        # bounded reproduction：从较优 depth 2 公式和 root genes 继续生成 depth 3。
        depth3_formulas = []
        parent_pool = [x["formula"] for x in population[: max(6, POPULATION_SIZE // 2)]] + gene_pool
        for _ in range(DEPTH3_SAMPLE_SIZE):
            a = rng.choice(parent_pool)
            b = rng.choice(source_pool)
            op = rng.choice(ops)
            child = binary(op, a, b)
            # replacement 思想：保留围绕强 parent 产生的子代，但最终仍由 IC 和 QD 决定是否进入记录。
            if formula_name(a) != formula_name(b):
                depth3_formulas.append(child)
        depth3_scores = [evaluate(f) for f in depth3_formulas]

        all_scores = depth1_scores + warm_scores + depth3_scores
        all_scores = sorted(all_scores, key=lambda x: x["score"], reverse=True)
        return all_scores

    def select_diverse_alphas(candidates, hist_panel, train_mask):
        train_dates = sorted(pd.to_datetime(hist_panel.loc[train_mask, "date"]).dropna().unique())
        if len(train_dates) > 90:
            step = max(1, len(train_dates) // 90)
            sample_dates = list(train_dates[::step])
        else:
            sample_dates = train_dates

        selected = []
        signatures = []
        # 先放入稳定基线，避免搜索结果在弱验证年完全失效。
        for cand in candidates:
            if "linear(" in cand["name"]:
                sig = first_pc_signature(cand["formula"], hist_panel, sample_dates)
                selected.append(cand)
                signatures.append(sig)
                break

        for cand in candidates:
            if len(selected) >= MAX_SELECTED_ALPHAS:
                break
            if cand["train_ic"] < MIN_TRAIN_IC or cand["valid_ic"] < MIN_VALID_IC:
                continue
            if any(cand["name"] == item["name"] for item in selected):
                continue
            sig = first_pc_signature(cand["formula"], hist_panel, sample_dates)
            too_similar = False
            for old_sig in signatures:
                if abs(corr_signature(sig, old_sig)) > PCA_SIM_THRESHOLD:
                    too_similar = True
                    break
            if too_similar:
                continue
            selected.append(cand)
            signatures.append(sig)
        return selected

    def formula_matrix(formulas, df):
        cols = []
        for item in formulas:
            raw_values = eval_formula(item["formula"], df)
            x = clean_factor_values(raw_values, df["date"])
            cols.append(x.to_numpy(dtype=np.float64, copy=True))
        if not cols:
            return np.zeros((len(df), 0), dtype=np.float64)
        mat = np.vstack(cols).T
        return np.nan_to_num(mat, nan=0.0, posinf=0.0, neginf=0.0)

    def mean_ic_from_array(values, panel, mask):
        x = pd.Series(values, index=panel.index)
        x = clean_factor_values(x, panel["date"])
        return fast_mean_ic(x, panel["label_cs"], panel["date"], mask)

    def choose_anchor_signal(hist_panel, pred_panel, valid_mask):
        """在验证年选择表现最稳的 anchor，并尝试两个 anchor 的小型组合。"""
        records = []
        for name, formula in anchor_formula_library():
            try:
                hist_signal = clean_factor_values(eval_formula(formula, hist_panel), hist_panel["date"])
                pred_signal = clean_factor_values(eval_formula(formula, pred_panel), pred_panel["date"])
                score = mean_ic_from_array(hist_signal.to_numpy(dtype=np.float64, copy=True), hist_panel, valid_mask)
            except Exception:
                continue
            if np.isfinite(score):
                records.append(
                    {
                        "name": name,
                        "hist": hist_signal,
                        "pred": pred_signal,
                        "score": score,
                    }
                )

        if not records:
            hist_signal = stable_base_signal(hist_panel)
            pred_signal = stable_base_signal(pred_panel)
            score = mean_ic_from_array(hist_signal.to_numpy(dtype=np.float64, copy=True), hist_panel, valid_mask)
            return hist_signal, pred_signal, score

        records = sorted(records, key=lambda x: x["score"], reverse=True)
        stable_record = None
        for record in records:
            if record["name"] == "stable_book":
                stable_record = record
                break
        best_hist = records[0]["hist"]
        best_pred = records[0]["pred"]
        best_score = records[0]["score"]

        top_records = records[: min(4, len(records))]
        for i in range(len(top_records)):
            for j in range(i + 1, len(top_records)):
                for weight in [0.25, 0.50, 0.75]:
                    hist_signal = weight * top_records[i]["hist"] + (1.0 - weight) * top_records[j]["hist"]
                    pred_signal = weight * top_records[i]["pred"] + (1.0 - weight) * top_records[j]["pred"]
                    score = mean_ic_from_array(hist_signal.to_numpy(dtype=np.float64, copy=True), hist_panel, valid_mask)
                    if score > best_score + 0.0003:
                        best_hist = hist_signal
                        best_pred = pred_signal
                        best_score = score

        if stable_record is not None and best_score < stable_record["score"] + MIN_ANCHOR_IMPROVEMENT:
            return stable_record["hist"], stable_record["pred"], stable_record["score"]

        return best_hist, best_pred, best_score

    def run_autoalpha():
        test_start = pd.to_datetime(start_date).normalize()
        test_end = pd.to_datetime(end_date).normalize()
        valid_start = pd.Timestamp(year=test_start.year - 1, month=1, day=1)
        valid_end = pd.Timestamp(year=test_start.year - 1, month=12, day=31)
        train_start = pd.Timestamp("2019-01-01")
        train_end = valid_start - pd.Timedelta(days=1)

        pred_panel = load_daily_panel(test_start, test_end)
        if pred_panel.empty:
            return pd.DataFrame(columns=["date", "instrument", "factor"])
        if "f_avg_depth_imbalance" not in pred_panel.columns or "f_avg_rel_spread" not in pred_panel.columns:
            pred_panel["factor"] = 0.0
            return finalize_factor(pred_panel)
        pred_panel = merge_external_features(pred_panel, test_start, test_end)
        pred_base_signal = stable_base_signal(pred_panel)

        feature_cols = signal_columns(pred_panel)
        if test_start.year <= 2020 or train_end < train_start + pd.Timedelta(days=240):
            pred_panel["factor"] = pred_base_signal
            return finalize_factor(pred_panel)

        hist_panel = load_daily_panel(train_start, valid_end)
        if hist_panel.empty:
            pred_panel["factor"] = pred_base_signal
            return finalize_factor(pred_panel)
        if "f_avg_depth_imbalance" not in hist_panel.columns or "f_avg_rel_spread" not in hist_panel.columns:
            pred_panel["factor"] = pred_base_signal
            return finalize_factor(pred_panel)
        hist_panel = merge_external_features(hist_panel, train_start, valid_end)

        feature_cols = signal_columns(hist_panel)
        train_mask = (
            (hist_panel["date"] >= train_start)
            & (hist_panel["date"] <= train_end)
            & hist_panel["label_cs"].notna()
        )
        valid_mask = (
            (hist_panel["date"] >= valid_start)
            & (hist_panel["date"] <= valid_end)
            & hist_panel["label_cs"].notna()
        )
        if int(train_mask.sum()) < 50000 or int(valid_mask.sum()) < 10000:
            pred_panel["factor"] = pred_base_signal
            return finalize_factor(pred_panel)

        hist_base_signal, pred_base_signal, base_score = choose_anchor_signal(hist_panel, pred_panel, valid_mask)

        candidates = evaluate_candidates(hist_panel, feature_cols, train_mask, valid_mask)
        selected = select_diverse_alphas(candidates, hist_panel, train_mask)
        if not selected:
            pred_panel["factor"] = pred_base_signal
            return finalize_factor(pred_panel)

        x_hist = formula_matrix(selected, hist_panel)
        x_pred = formula_matrix(selected, pred_panel)
        y = hist_panel["label_cs"].to_numpy(dtype=np.float64, copy=True)
        train_idx = train_mask.to_numpy()
        valid_idx = valid_mask.to_numpy()

        # AutoAlpha 论文最后用 learning-to-rank 模型；这里用 Ridge 作为轻量 ranker。
        # 关键保护：AutoAlpha 信号必须在验证年明显超过稳定基线，否则直接回退。
        best_score = mean_ic_from_array(hist_base_signal.to_numpy(dtype=np.float64, copy=True), hist_panel, valid_mask)
        best_kind = "anchor"
        best_coef = None
        best_blend = 0.0

        def consider_signal(signal, coef=None, kind="weighted"):
            nonlocal best_score, best_kind, best_coef, best_blend
            signal = np.asarray(signal, dtype=np.float64)
            for direction in [1.0, -1.0]:
                oriented = direction * signal
                for blend in [0.04, 0.08, 0.12, 0.16, 0.20, 0.25, 0.30, 0.35]:
                    candidate = (1.0 - blend) * hist_base_signal.to_numpy(dtype=np.float64, copy=True) + blend * oriented
                    score = mean_ic_from_array(candidate, hist_panel, valid_mask)
                    if score > best_score + 0.0005:
                        best_score = score
                        best_kind = kind
                        best_coef = None if coef is None else direction * coef
                        best_blend = blend

        base_weights = np.array([max(item["valid_ic"], 0.0) + 0.001 for item in selected], dtype=np.float64)
        base_weights = base_weights / (base_weights.sum() + EPS)
        weighted_signal = x_hist @ base_weights
        consider_signal(weighted_signal, coef=base_weights, kind="weighted")

        x_train = x_hist[train_idx]
        y_train = y[train_idx]
        mask_train = np.isfinite(y_train)
        x_train = x_train[mask_train]
        y_train = y_train[mask_train]
        if len(y_train) >= 50000 and x_train.shape[1] > 0:
            xtx = x_train.T @ x_train
            xty = x_train.T @ y_train
            identity = np.eye(x_train.shape[1], dtype=np.float64)
            for alpha in [0.1, 1.0, 3.0, 10.0, 30.0, 100.0]:
                try:
                    coef = np.linalg.solve(xtx + alpha * identity, xty)
                except Exception:
                    continue
                consider_signal(x_hist @ coef, coef=coef, kind="ridge")

        if best_kind == "anchor" or best_score < base_score + MIN_AUTOALPHA_IMPROVEMENT:
            pred_panel["factor"] = pred_base_signal
            return finalize_factor(pred_panel)

        if best_kind in ["ridge", "weighted"] and best_coef is not None:
            auto_signal = x_pred @ best_coef
            pred_panel["factor"] = (1.0 - best_blend) * pred_base_signal + best_blend * auto_signal
        else:
            pred_panel["factor"] = pred_base_signal
        return finalize_factor(pred_panel)

    try:
        result = run_autoalpha()
        if result.empty:
            pool = read_stock_pool(start_date, end_date)
            pool["factor"] = 0.0
            return finalize_factor(pool)
        return result[["date", "instrument", "factor"]]
    except Exception:
        # 提交平台不展示完整 traceback。兜底返回一个稳定的简化盘口因子，避免直接失败。
        try:
            panel = load_daily_panel(start_date, end_date)
            if panel.empty:
                pool = read_stock_pool(start_date, end_date)
                pool["factor"] = 0.0
                return finalize_factor(pool)
            if "f_avg_depth_imbalance" not in panel.columns or "f_avg_rel_spread" not in panel.columns:
                panel["factor"] = 0.0
                return finalize_factor(panel)
            panel["factor"] = stable_base_signal(panel)
            return finalize_factor(panel)
        except Exception:
            pool = read_stock_pool(start_date, end_date)
            pool["factor"] = 0.0
            return finalize_factor(pool)
